# 0077: the read on dev_real (arm R and arm L)

The answers only, free values, one decoder. Each of dev_real's 300 instructions (an OASST1 prompt with 2 to 8 checks
inside it) is read whole, its answer held token by token to the grammar of its own instruction. Dev is for looking;
nothing is decided on it.

It writes `rules/reads/0077/view_real_{r,l}_dev.jsonl` (resumable). **Disconnect when it prints `read: dev`.**

In [ ]:
import hashlib, json, os, signal, subprocess, sys, time
BRANCH = 'master'
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q -b {BRANCH} https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git fetch -q && git checkout -q {BRANCH} && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'

PHASE = 'dev'
ROWS     = f'{DRIVE}/rules/long_rows_0077/read_rows.jsonl'
ROWS74   = f'{DRIVE}/rules/long_rows_0074/read_rows.jsonl'
ADAPTERS = {'r': f'{DRIVE}/rules/cubby450m_view_real_r', 'l': f'{DRIVE}/rules/cubby450m_view_long_l'}
READS    = f'{DRIVE}/rules/reads/0077'
for p in [CKPT, TOKENIZER, ROWS, ROWS74] + [f'{d}/{x}' for d in ADAPTERS.values() for x in ('talk_lora.safetensors', 'talk_lora.json', 'DONE')]:
    assert os.path.exists(p), p
mans = {}
for rows_p, name in ((ROWS, '0077'), (ROWS74, '0074')):
    mans[name] = json.load(open(f'{REPO}/validation/logs/long_rows_{name}_read_rows.manifest.json'))
    assert hashlib.sha256(open(rows_p, 'rb').read()).hexdigest() == mans[name]['sha256'], f'{rows_p} is not the committed file'
    keys = {k for ln in open(rows_p, encoding='utf-8') for k in json.loads(ln)}
    assert keys == {'id', 'split', 'instruction'}, f'the rows file holds more than ids and instructions: {keys}'
    print(name, mans[name]['rows'], 'rows', mans[name]['by_split'])
os.makedirs(READS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
PASSES = [(arm, ROWS, 'dev_real', f'{READS}/view_real_{arm}_dev.jsonl', mans['0077']) for arm in 'rl']

os.environ['CB_NO_FLEX'] = '1'
N = 300
for arm, rows_p, splits, OUT, man in PASSES:
    want = sum(min(N, man['by_split'][s]) for s in splits.split(','))
    have = [json.loads(ln) for ln in open(OUT, encoding='utf-8')] if os.path.exists(OUT) else []
    assert not any(d.get('own_text') for d in have), f'{OUT} was read with the own-text rule: move it aside'
    if len({d['id'] for d in have}) >= want:
        print(f'--- arm {arm}: {OUT} is whole ({len(have)} answers): not read again')
        continue
    print(f'--- arm {arm}: {splits} -> {OUT}')
    cmd = [sys.executable, '-u', 'validation/exp_view_split_torch.py', '--arm', 'm', '--ckpt', CKPT, '--tokenizer', TOKENIZER,
           '--adapter', ADAPTERS[arm], '--rows', rows_p, '--splits', splits, '--n', str(N), '--batch', '32', '--out', OUT]
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'the read failed ({p.returncode})'
    !wc -l "{OUT}"
print('read:', PHASE, '(all passes read)')

## Done

**Disconnect the runtime** (on `read: dev`). At home dev_real is scored on the VM for both arms and the two reports are committed; then `view_real_read_test.ipynb` reads the held-out sets once.